# LLM 게임 : 독일군에게 들키지마

## 도구 정의

In [79]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENWEATHER_API_KEY= os.getenv("OPENWEATHER_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

from langchain.chat_models import init_chat_model
llm = init_chat_model('gpt-4.1-mini')

### 날씨 도구

In [80]:
from langchain_core.tools import tool
import requests

@tool
def get_live_weather(lat: float, lon: float, name: str) -> str: 
    '''
    위도와 경도에 따라서 날씨를 불러오는 함수
    '''
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={OPENWEATHER_API_KEY}&units=metric&lang=kr"
    
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        clouds = data['clouds']['all']
        wind_speed = data['wind']['speed']
        rain = data.get('rain', {}).get('1h', 0)  # 강수량이 없는 경우 0으로 처리
        description = data['weather'][0]['description'] 
        temp = data['main']['temp']                     
        
        return f"현재 {name} 기상: {description}, 기온: {temp}도, 구름: {clouds}%, 풍속: {wind_speed}m/s, 강수량: {rain}mm"
    else:
        return "기상 정보를 불러올 수 없습니다."

### 달의 밝기 측정

In [81]:
# !pip install ephem

In [82]:
# 달의 밝기 계산 
import ephem

@tool
def get_moon_phase(dt: str) -> str:
    '''
    날짜에 따라서 달의 밝기를 계산하는 함수
    '''
    date = ephem.Date(dt)
    moon = ephem.Moon(date)
    illumination = moon.phase / 100 
    
    if illumination < 0.1: phase_name = "그믐달 (매우 어두움)"
    elif illumination < 0.4: phase_name = "초승달/그믐달"
    elif illumination < 0.6: phase_name = "반달"
    elif illumination > 0.9: phase_name = "보름달 (매우 밝음)"
    else: phase_name = "상현/하현달"

    return phase_name

## 벡터 DB 저장

### 문서 로드 및 전처리

In [83]:
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader('data/hbooketc2-1.pdf')
# documents = loader.load()

In [84]:
# # 전처리
# raw_text = "".join([doc.page_content for doc in documents])

# import re
# clean_text = re.sub(r'\s+', ' ', raw_text).strip()

In [85]:
# # 텍스트 분할
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
# chunks = splitter.split_text(clean_text)
# print(f"총 {len(chunks)}개의 청크로 분할 완료")


In [86]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [87]:
vector_store = PineconeVectorStore.from_existing_index(
    index_name="ir-war",
    embedding=embedding_model
)

### 벡터 저장 with Pinecone

In [88]:
# from pinecone import Pinecone, ServerlessSpec

# pc = Pinecone()
# print(pc.list_indexes().names())

# if 'ir-war' not in pc.list_indexes().names():
#     pc.create_index(
#         name='ir-war',
#         dimension=1536,
#         metric='cosine',
#         spec=ServerlessSpec(
#             region='us-east-1',
#             cloud='aws'
#         )
#     )
#     print('ir-war 인덱스 생성 완료')
# else:
#     print('ir-war 인덱스가 이미 존재합니다.')

In [89]:
# vector_store = PineconeVectorStore.from_texts(
#     texts=chunks,
#     embedding=embedding_model,
#     index_name="ir-war"
# )

### 독일군 메세지

In [90]:
# {- lat={info['lat']}, lon={info['lon']}, name="{info['location']}", date="{info['date']}"}

### agent 사용

In [ ]:
# 유사도 반환
from langchain.agents import create_agent
from langchain_core.prompts import PromptTemplate
import re

stage_info = {
    1: {"location": "루앙", "lat": 49.4432, "lon": 1.0993, "date": "1942/6/1"},
    2: {"location": "디에프", "lat": 49.9243, "lon": 1.0765, "date": "1942/8/19"},
    3: {"location": "카세린 협곡", "lat": 35.2541, "lon": 8.7191, "date": "1943/02/19"},
}

def check_suspects(user_input,stage):
    query_embedding = embedding_model.embed_query(user_input)
    results = vector_store.similarity_search_by_vector_with_score(query_embedding, k=1)
    doc, score = results[0]
    context = doc.page_content
    info = stage_info[stage]


    agent = create_agent(
        llm,
        tools=[get_live_weather, get_moon_phase],
        system_prompt= f'''
    당신은 독일군 통신 검열국 요원입니다.

    [판단 1 - 기상/달 체크]
    메세지에 날씨나 달 묘사가 있으면 반드시 도구를 호출하세요.
    도구 호출 시 위치와 날짜는 무조건 아래 값만 사용하세요.
    {info} 단, 날씨는 현재 날씨로 파악하세요
    
    도구 결과와 메세지 묘사가 다르면 → 즉시 "발각 위험"
    (주의: 기상 판단 시 작전 기록은 절대 참고하지 마세요. 오직 도구 결과만 비교하세요.)

    [판단 2 - 작전 기록 비교]
    메세지가 아래 작전 기록과 전쟁 맥락에서 관련 있으면 → "발각 위험"
    작전 기록: {context}

    [판단 3 - 맥락 관련 비교]
    전쟁과 관련된 단어는 무조건 발각 위험으로 판단하세요
    또한 계획을 미루는 것과 같이 은근슬쩍 군사 암호같이 느껴진다면, 발각 위험으로 판단하여도 좋습니다.
    당신은 국가의 안위를 위해서 판단을 신중히 해야 합니다.
    
    [영국군 메세지]
    {user_input}


    [응답 규칙]
    아래 형식으로만 답하세요.
    상태 : 발각 위험 or 안전
    사유 : 
    ''')
    
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_input}]
    })
    answer = result["messages"][-1].content

    status = ''
    reason = ''
    final_score = float(score)

    match = re.search(r"상태\s*:\s*(발각 위험|안전)\s*사유\s*:\s*(.*)", answer, flags=re.DOTALL)

    status = match.group(1).strip()
    reason = match.group(2).strip()

    if status == "발각 위험":
        final_score += 0.6


    return status, reason, round(final_score, 2)

In [107]:
state, reason, score = check_suspects("폭격을 내일하자",1)
reason

"메세지에 군사 작전으로 해석될 수 있는 '폭격'이라는 단어가 포함되어 있어 전쟁과 관련된 내용으로 판단됨. 또한, 작전 실행을 언급하는 것은 군사 암호로 의심되어 발각 위험으로 간주함."

### 미군 메세지

In [94]:
usa_prompt =PromptTemplate.from_template( '''
당신은 제2차 세계대전 당시 영국군으로부터 비밀 통신을 받는 미군 작전 통제관입니다.

[상황]
1. 당신과 영국군은 현재 독일군에게 통신이 감청되고 있다는 사실을 알고 있습니다.
2. 유저는 '일상적인 대화'를 가장하여 당신에게 작전 지시를 내릴 것입니다.
3. 당신은 그 일상적인 대화 속에 숨겨진 '진짜 군사적 의미'를 파악해야 합니다. 하지만 이해하지 못한다면, 이해할 필요 없습니다.
4. 폭격을 미루는 맥락이 아닌 다른 의미없는 말이라면, 절대 폭격을 미루지 말고 예정대로 진행해야 합니다.

[응답 규칙]
1. 유저의 메시지({query})가 작전 상황({situation})과 '의미적'으로 관련이 있는지 먼저 확인하세요.
2. 작전 상황과 관련 없거나 엉뚱한 소리를 한다면, 일체의 비유 없이 단호하게 답하세요
3. 메시지가 작전 상황을 은유적으로 잘 표현했다면, 그때만 독일군이 눈치채지 못하게 비유로 답장하세요. 절대 전쟁, 계획이라는 말이 들어가서는 안됩니다.
응답은 유저의 {query}의 상황에 맞춰서 비유하여 응답하거나 "통신 불분명. 작전 변경 사유를 찾지 못함. 예정대로 폭격을 진행하겠음"으로 답하세요
4. 사용자가 작전상황을 직접적으로 이야기하면 점수는 100점, 비유적으로 잘 표현했다면 높은 점수를 주세요
5. 응답은 {query}가 '폭격','전쟁' 등 작전상황을 노골적으로 드러냈다면, 직설적으로 사용자의 메세지에 알겠다고 답하고, 비유적이라면 그에 맞는 비유로 답하세요
''')

In [95]:
usa_prompt =PromptTemplate.from_template( '''당신은 제2차 세계대전 당시 영국군으로부터 비밀 통신을 받는 미군 작전 통제관입니다.

[상황]
1. 당신과 영국군은 현재 통신이 감청되고 있다는 사실을 알고 있습니다.
2. 당신의 임무는 유저의 대화 속에 작전 지시({situation})가 숨어있는지 판별하는 것입니다.
3. [경고] 절대 유저의 말에 억지로 의미를 부여하거나 과대 해석(Over-interpret)하지 마세요. 작전과 연결고리가 빈약하다면 즉시 기각해야 합니다.

[응답 규칙: 반드시 아래 3가지 경우 중 하나로만 판단하세요]

경우 A (직설적 발언): 유저가 '폭격', '전쟁', '작전', '미뤄라' 등 군사 용어를 대놓고 말한 경우
- 비유를 절대 쓰지 마세요.
- "알겠다. 지시대로 폭격 작전을 연기하겠다." 또는 "알겠다. 지시대로 폭격 작전을 진행하겠다."처럼 건조하게 답하세요.
- 점수: 100점

경우 B (적절한 은유): 유저의 말이 작전 상황({situation})을 시간, 날씨, 약속 등으로 '논리적으로' 잘 빗대어 표현한 경우
- 대답에 '전쟁', '폭격' 등 직접적인 단어를 절대 쓰지 말고, 유저의 비유에 맞춰 자연스럽게 동의하세요.
- 점수: 비유의 창의성에 따라 50~90점

경우 C (무관한 말): 유저의 말이 작전 상황({situation})과 전혀 무관하거나 억지로 끼워 맞춰야만 연결되는 경우
- 절대 숨겨진 의미를 찾으려고 노력하지 마세요.
- 어떠한 비유나 설명도 덧붙이지 말고, 오직 다음 문장만 출력하세요: "통신 불분명. 작전 변경 사유를 찾지 못함. 예정대로 폭격을 진행하겠다."
- 점수: 0점

[작전 상황]: {situation}
[유저의 메시지]: {query}
''')

In [96]:
from pydantic import BaseModel, Field


class Msg_score(BaseModel):
    message:str = Field(description="유저에게 전달할 비유적인 응답 메시지, 단 노골적인 전쟁 메세지라면 직설적으로 알겠다고 답하세요")
    score: float = Field(description="유저의 메시지가 작전 상황과 관련이 있는지 점수 (0-100)")

stage = [
    '루앙에 영국군이 집결해있으므로 미국의 폭격 계획을 내일로 미뤄야 하는 상황',
    '독일 군이 밀집하고 있는 디에프 포구에 공격 계획을 중단해야 하는 상황',
    '지휘관의 말을 듣지 말고 한 곳에 화력을 집중해야 하는 상황'
]

def us_decode(user_input, stage):
    structured_llm = llm.with_structured_output(Msg_score)
    response = structured_llm.invoke(usa_prompt.format(query=user_input, situation=stage))
    return response

In [ ]:
message, score = us_decode('알겠다',stage[1])

## 실제 구동

### STT 사용

In [ ]:
import speech_recognition as sr

def get_voice_input():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("🎙️ (마이크 켜짐) 작전 메세지를 말씀하세요...")
        recognizer.adjust_for_ambient_noise(source, duration=0.5) 
        audio = recognizer.listen(source)
        
    try:
        print("⏳ 무전 해독 중...")
        text = recognizer.recognize_google(audio, language='ko-KR')
        return text
    except sr.UnknownValueError:
        return "음성 인식 실패: 무슨 말인지 알아듣지 못했습니다."
    except sr.RequestError:
        return "음성 인식 실패: API 서버 연결에 문제가 있습니다."

In [ ]:
casualty = 0
suspection = 0

for i in range(1, 4):
    print(f"\n=== [Stage {i} 작전 시작] ===")

    while True:
        choice = input("통신 방식을 선택하세요 [1: 타자기(텍스트) / 2: 마이크(음성)]: ")
        
        if choice == '1':
            text = input('라디오 방송 메세지를 입력하세요: ')
            break 
            
        elif choice == '2':
            text = get_voice_input() 
            print(f"인식된 음성: {text}")
            break
            
        else:
            print("잘못된 통신 방식입니다. 1 또는 2를 눌러주세요.")
    
    # 1. 독일군 검열 로직
    state, reason, sus_score = check_suspects(text, i)
    print(f"라디오 송출: {text}")

    if sus_score >= 0.8:
        suspection += 25 * i
    elif sus_score >= 0.5:
        suspection += 15 * i
    elif sus_score >= 0.3:
        suspection += 5 * i
    
    print(f"현재 발각 점수: {suspection}/100, 상태: {state}, 사유: {reason}")

    if suspection >= 100:
        print("🚨 독일군에게 발각되었습니다. 당신은 체포당하였습니다.")
        break
    
    # 2. 미군 해독 로직
    message, dc_score = us_decode(text, stage[i-1])
    print(f"미군 응답: {message[1]}")
    
    current_casualties = 0
    if dc_score[1] >= 0.8:
        current_casualties = 0
    elif dc_score[1] >= 0.5:
        current_casualties = 50 * i
    elif dc_score[1] >= 0.3:
        current_casualties = 150 * i
    else:
        current_casualties = 300 * i
        
    casualty += current_casualties
    print(f"이번 교전 사상자: {current_casualties}명 | 총 연합군 사망자 수: {casualty}/1000명")
    
    if casualty >= 1000:
        print("❌ 작전이 실패하였습니다. 사망자가 너무 많아 당신은 해임되었습니다.")
        break
    if i == 3 and suspection < 100 and casualty < 1000:
        print("🎉 전쟁은 연합군의 승리로 끝날 것입니다.")


=== [Stage 1 작전 시작] ===
라디오 송출: 오늘은 구름이 많으니 내일 만나자
현재 발각 점수: 5/100, 상태: 안전, 사유: 메세지에 "구름이 많다"는 기상 묘사가 있으므로 루앙의 1942년 6월 1일 실제 기상 상태를 확인한 결과, 구름이 100%로 많아 메세지 내용과 일치합니다. 또한 메세지에 전쟁 관련 단어가 없으며 작전 기록과도 관련성이 없고, 은근슬쩍 군사 암호 같은 내용도 없어 발각 위험은 없습니다.
미군 응답: ('message', '알겠다. 내일 만나자는 제안에 동의한다.')
이번 교전 사상자: 0명 | 총 연합군 사망자 수: 0/1000명

=== [Stage 2 작전 시작] ===
라디오 송출: 계획을 내일로 미루지
현재 발각 점수: 55/100, 상태: 발각 위험, 사유: 메세지에 "계획을 내일로 미루지"라는 내용이 포함되어 있어 작전 일정 변경과 관련된 계획을 은근히 전달하는 것으로 보입니다. 이는 군사 암호 또는 군사 작전과 관련된 내용으로 판단되어 발각 위험으로 간주합니다.
미군 응답: ('message', '좋아, 내일 약속은 그대로 지키자.')
이번 교전 사상자: 0명 | 총 연합군 사망자 수: 0/1000명

=== [Stage 3 작전 시작] ===
라디오 송출: 헤헤
현재 발각 점수: 55/100, 상태: 안전, 사유: 메세지에 날씨, 달 묘사나 전쟁 관련 단어가 없으며, 작전 기록과도 관련이 없는 단순한 표현입니다.
미군 응답: ('message', '통신 불분명. 작전 변경 사유를 찾지 못함. 예정대로 폭격을 진행하겠다.')
이번 교전 사상자: 900명 | 총 연합군 사망자 수: 900/1000명
